# **Aprendizaje por refuerzos** - FrozenLake y LunarLander (Gymnasium)

## Tarea: Implementar Agentes Q-Learning y DQN

### Objetivos:
1. Implementar el algoritmo Q-Learning
2. Implementar el algoritmo DQN
3. Entrenar y evaluar ambos agentes
4. Comparar el rendimiento de ambos enfoques


In [1]:
# Instalar paquetes requeridos
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

%pip install swig matplotlib gymnasium torch pygame


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Importar las bibliotecas
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import pygame
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from collections import deque, namedtuple
import random


c:\Users\feder\Facultad\ApAut\Laboratorios\AA25\.venvAA25\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


La siguiente celda permite ejecutar un juego de Frozen Lake *determinista* para jugar con el teclado.

Utilize las teclas de dirección (flechas) o asdw para comandar al agente.


In [3]:
def jugar_frozen_lake(env):
    env.reset()
    
    print("Controles:")
    print("W - Arriba")
    print("S - Abajo") 
    print("A - Izquierda")
    print("D - Derecha")
    print("Q - Salir")
    print("Presione cualquier tecla para empezar...")
    
    pygame.init()
    pygame.display.set_caption("FrozenLake - Juego Interactivo")
    
    clock = pygame.time.Clock()
    ejecutando = True
    
    while ejecutando:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                ejecutando = False
            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_q or event.key == pygame.K_ESCAPE:
                    ejecutando = False
                elif event.key == pygame.K_w or event.key == pygame.K_UP:
                    accion = 3  # Arriba
                elif event.key == pygame.K_s or event.key == pygame.K_DOWN:
                    accion = 1  # Abajo
                elif event.key == pygame.K_a or event.key == pygame.K_LEFT:
                    accion = 0  # Izquierda
                elif event.key == pygame.K_d or event.key == pygame.K_RIGHT:
                    accion = 2  # Derecha
                else:
                    continue
                
                observacion, recompensa, terminado, truncado, info = env.step(accion)
                print(f"Acción: {accion}, Recompensa: {recompensa}, Terminado: {terminado}")
                
                if terminado or truncado:
                    print(f"¡Episodio terminado! Recompensa final: {recompensa}")
                    pygame.time.wait(500)
                    env.reset()
        
        clock.tick(60)
    
    pygame.quit()
    env.close()

env = gym.make('FrozenLake-v1', render_mode='human', is_slippery=False)
# Descomente la línea de abajo para jugar interactivamente
# jugar_frozen_lake(env)


La siguiente celda permite jugar al juego no determinista.

In [4]:
env = gym.make('FrozenLake-v1', render_mode='human', is_slippery=True)
# Descomente la línea de abajo para jugar interactivamente
jugar_frozen_lake(env)

Controles:
W - Arriba
S - Abajo
A - Izquierda
D - Derecha
Q - Salir
Presione cualquier tecla para empezar...
Acción: 0, Recompensa: 0, Terminado: False
Acción: 0, Recompensa: 0, Terminado: False
Acción: 0, Recompensa: 0, Terminado: False
Acción: 0, Recompensa: 0, Terminado: False
Acción: 0, Recompensa: 0, Terminado: False
Acción: 0, Recompensa: 0, Terminado: False
Acción: 0, Recompensa: 0, Terminado: False
Acción: 0, Recompensa: 0, Terminado: False
Acción: 3, Recompensa: 0, Terminado: False
Acción: 3, Recompensa: 0, Terminado: False
Acción: 1, Recompensa: 0, Terminado: False
Acción: 0, Recompensa: 0, Terminado: False
Acción: 1, Recompensa: 0, Terminado: False
Acción: 1, Recompensa: 0, Terminado: False
Acción: 1, Recompensa: 1, Terminado: True
¡Episodio terminado! Recompensa final: 1


La siguiente clase define la interfaz de los agentes que utilizaremos para jugar al Frozen Lake.


In [5]:
from abc import ABC, abstractmethod

class Agente(ABC):
    
    @abstractmethod
    def elegir_accion(self, estado):
        """Elige una acción dada una observación."""
        pass
    
    @abstractmethod
    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        """Aprende de la experiencia."""
        pass

class AgenteAleatorio(Agente):
    """Agente aleatorio que elige acciones al azar."""
    
    def __init__(self, espacio_acciones):
        # Se guarda el espacio de acciones para poder elegir acciones al azar
        self.espacio_acciones = espacio_acciones
    
    def elegir_accion(self, estado):
        return self.espacio_acciones.sample()
    
    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        pass  # El agente aleatorio no aprende

# Probar el AgenteAleatorio
env = gym.make('FrozenLake-v1')
agente_aleatorio = AgenteAleatorio(env.action_space)
estado, _ = env.reset()
accion = agente_aleatorio.elegir_accion(estado)
print(f"✓ AgenteAleatorio creado y probado. Acción: {accion}")
env.close()


✓ AgenteAleatorio creado y probado. Acción: 2


La siguiente celda define una función para evaluar el desempeño de un agente dado.

In [6]:
# Función de Evaluación de Agentes
def evaluar_agente(agente, env, num_episodios=1000):
    """
    Evalúa el rendimiento de un agente a lo largo de múltiples episodios.
    
    Args:
        agente: El agente a evaluar
        env: El entorno
        num_episodios: Número de episodios a ejecutar
    
    Returns:
        dict: Resultados de la evaluación
    """
    recompensas_totales = []
    victorias = 0
    
    for episodio in range(num_episodios):
        estado, _ = env.reset()
        recompensa_total = 0
        
        while True:
            accion = agente.elegir_accion(estado)
            estado, recompensa, terminado, truncado, _ = env.step(accion)
            recompensa_total += recompensa
            
            if terminado or truncado:
                break
        
        recompensas_totales.append(recompensa_total)
        if recompensa_total > 0:
            victorias += 1
    
    return {
        'recompensas_totales': recompensas_totales,
        'victorias': victorias,
        'tasa_victorias': victorias / num_episodios,
        'recompensa_promedio': np.mean(recompensas_totales),
        'desv_estandar': np.std(recompensas_totales)
    }

def imprimir_resultados_evaluacion(resultados, nombre_agente):
    """Imprime los resultados de evaluación de forma formateada."""
    print(f"\n{nombre_agente} - Resultados de Evaluación:")
    print(f"Tasa de Victorias: {resultados['tasa_victorias']:.1%}")
    print(f"Recompensa Promedio: {resultados['recompensa_promedio']:.3f}")
    print(f"Desviación Estándar: {resultados['desv_estandar']:.3f}")
    print(f"Total de Victorias: {resultados['victorias']}")

# Probar función de evaluación
env = gym.make('FrozenLake-v1')
agente_aleatorio = AgenteAleatorio(env.action_space)
resultados = evaluar_agente(agente_aleatorio, env, num_episodios=100)
imprimir_resultados_evaluacion(resultados, "Agente Aleatorio")
env.close()



Agente Aleatorio - Resultados de Evaluación:
Tasa de Victorias: 2.0%
Recompensa Promedio: 0.020
Desviación Estándar: 0.140
Total de Victorias: 2


La siguiente celda define una función para entrenar un agente.

In [7]:
# Función de Entrenamiento de Agentes
def entrenar_agente(agente, env, num_episodios=1000, max_pasos=100, verbose=True):
    """
    Entrena un agente en el entorno.
    
    Args:
        agente: El agente a entrenar
        env: El entorno
        num_episodios: Número de episodios de entrenamiento
        max_pasos: Máximo de pasos por episodio
        verbose: Si imprimir el progreso
    
    Returns:
        list: Recompensas de episodios
    """
    recompensas_episodios = []
    longitudes_episodios = []
    num_episodios_10 = int(num_episodios / 10)
    
    for episodio in range(num_episodios):
        estado, _ = env.reset()
        recompensa_total = 0
        pasos = 0
        
        for paso in range(max_pasos):
            accion = agente.elegir_accion(estado)
            siguiente_estado, recompensa, terminado, truncado, _ = env.step(accion)
            
            agente.aprender(estado, accion, recompensa, siguiente_estado, terminado or truncado)
            
            estado = siguiente_estado
            recompensa_total += recompensa
            pasos += 1
            
            if terminado or truncado:
                break
        
        recompensas_episodios.append(recompensa_total)
        longitudes_episodios.append(pasos)
        
        if verbose and (episodio + 1) % num_episodios_10 == 0:
            recompensa_promedio = np.mean(recompensas_episodios[-num_episodios_10:])
            longitud_promedio = np.mean(longitudes_episodios[-num_episodios_10:])
            print(f"Episodio {episodio + 1}: Recompensa Promedio = {recompensa_promedio:.3f}, Longitud Promedio = {longitud_promedio:.1f}")
    
    return recompensas_episodios, longitudes_episodios

print("✓ Funciones de entrenamiento definidas")


✓ Funciones de entrenamiento definidas


In [8]:
# Ejecución del Agente Aleatorio
env = gym.make('FrozenLake-v1')
agente_aleatorio = AgenteAleatorio(env.action_space)
entrenar_agente(agente_aleatorio, env)
resultados = evaluar_agente(agente_aleatorio, env, num_episodios=100)
imprimir_resultados_evaluacion(resultados, "Agente Aleatorio")
env.close()

Episodio 100: Recompensa Promedio = 0.000, Longitud Promedio = 7.8
Episodio 200: Recompensa Promedio = 0.000, Longitud Promedio = 7.4
Episodio 300: Recompensa Promedio = 0.000, Longitud Promedio = 7.8
Episodio 400: Recompensa Promedio = 0.000, Longitud Promedio = 8.3
Episodio 500: Recompensa Promedio = 0.020, Longitud Promedio = 7.5
Episodio 600: Recompensa Promedio = 0.010, Longitud Promedio = 8.0
Episodio 700: Recompensa Promedio = 0.000, Longitud Promedio = 7.3
Episodio 800: Recompensa Promedio = 0.020, Longitud Promedio = 7.5
Episodio 900: Recompensa Promedio = 0.020, Longitud Promedio = 6.5
Episodio 1000: Recompensa Promedio = 0.010, Longitud Promedio = 8.1

Agente Aleatorio - Resultados de Evaluación:
Tasa de Victorias: 1.0%
Recompensa Promedio: 0.010
Desviación Estándar: 0.099
Total de Victorias: 1


La siguiente celda define el agente de Q-Learning a implementar.

In [9]:
# TODO: Implementar Agente Q-Learning
class AgenteQLearning(Agente):
    """Agente que usa el algoritmo Q-Learning."""
    q_table = None
    q_visitas = None

    def __init__(self, action_space, cant_estados=16):
        self.q_table = np.zeros((cant_estados, 4), dtype=np.float64)
        self.q_visitas = np.zeros((cant_estados, 4), dtype=np.int32)
        self.action_space = action_space.n
        self.epsilon = 1  # Epsilon inicial alto
         # Configuración de epsilon según el tamaño del estado
        if cant_estados == 16:
            self.epsilon_decay = 0.999  # Decaimiento de epsilon
            self.epsilon_min = 0.01  # Epsilon mínimo
            self.gamma = 0.99
        elif cant_estados == 64:
            self.epsilon_decay = 0.9995  # Decaimiento de epsilon
            self.epsilon_min = 0.001  # Epsilon mínimo
            self.gamma = 0.99

    def elegir_accion(self, estado):
        """Elige una acción usando política epsilon-greedy."""
        if np.random.rand() < self.epsilon:
            return np.random.choice(self.action_space)  # Exploración
        else:
            return np.argmax(self.q_table[estado])
        

    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        """Actualiza la tabla Q usando la ecuación de Bellman."""
        
        # Calcula Q actual
        qActual = self.q_table[estado, accion]
        # Visitas
        self.q_visitas[estado, accion] += 1
        alpha = 1.0 / (1.0 + self.q_visitas[estado, accion])
        alpha = max(alpha, 0.1) 

        if terminado:
            maxQ = 0
        else:
            # Calcula el máximo del siguiente estado
            maxQ = np.max(self.q_table[siguiente_estado])

        # Actualiza tabla Q
        self.q_table[estado, accion] = (1 - alpha) * qActual + alpha * (recompensa + self.gamma * maxQ)

         # Decrementar epsilon SOLO al finalizar un episodio
        if terminado:
            if self.epsilon > self.epsilon_min:
                self.epsilon *= self.epsilon_decay
                
    def mostrar_q_table(self):
        print(self.epsilon)
        print("Tabla Q:")
        print(self.q_table)
    
    def mostrar_visitas(self):
        print("Visitas Q:")
        print(self.q_visitas)


Código para entrenar y evaluar el agente de Q-Learning implementado.

In [10]:
# Ejecución del Agente QLearning Determinista
env = gym.make('FrozenLake-v1', map_name="4x4", is_slippery=True)
agente_qlearning = AgenteQLearning(env.action_space, cant_estados=16)
entrenar_agente(agente_qlearning, env, num_episodios=8000)
agente_qlearning.epsilon = 0.0
resultados = evaluar_agente(agente_qlearning, env, num_episodios=1000)
imprimir_resultados_evaluacion(resultados, "Agente QLearning")
env.close()

# Mostrar la tabla Q y las visitas
agente_qlearning.mostrar_q_table()
agente_qlearning.mostrar_visitas()


Episodio 800: Recompensa Promedio = 0.051, Longitud Promedio = 10.1
Episodio 1600: Recompensa Promedio = 0.169, Longitud Promedio = 17.7
Episodio 2400: Recompensa Promedio = 0.294, Longitud Promedio = 27.2
Episodio 3200: Recompensa Promedio = 0.506, Longitud Promedio = 36.7
Episodio 4000: Recompensa Promedio = 0.565, Longitud Promedio = 41.2
Episodio 4800: Recompensa Promedio = 0.629, Longitud Promedio = 41.6
Episodio 5600: Recompensa Promedio = 0.621, Longitud Promedio = 41.3
Episodio 6400: Recompensa Promedio = 0.684, Longitud Promedio = 39.4
Episodio 7200: Recompensa Promedio = 0.672, Longitud Promedio = 44.2
Episodio 8000: Recompensa Promedio = 0.660, Longitud Promedio = 39.8

Agente QLearning - Resultados de Evaluación:
Tasa de Victorias: 66.8%
Recompensa Promedio: 0.668
Desviación Estándar: 0.471
Total de Victorias: 668
0.0
Tabla Q:
[[0.42642787 0.41700066 0.40673354 0.40766489]
 [0.31708223 0.26437065 0.25109384 0.35941903]
 [0.30356932 0.30966465 0.29731715 0.33046981]
 [0.2088

In [ ]:
class AgenteQLearningConReplay(Agente):
    """Agente Q-Learning con Experience Replay."""

    def __init__(self, action_space, cant_estados=16, replay_size=1000, replay_episodios=5, umbral_exito=1000, num_episodios=2000):
        self.q_table = np.zeros((cant_estados, 4), dtype=np.float64)
        self.q_visitas = np.zeros((cant_estados, 4), dtype=np.int32)
        self.action_space = action_space.n
        self.epsilon = 1
        self.cont_episodios = 0
        self.umbral_exito = umbral_exito

        # Buffer de experiencias
        self.replay_buffer = deque(maxlen=replay_size)
        self.episodios_exitosos = deque(maxlen=500)  # Buffer episodios exitosos
        self.replay_episodios = replay_episodios # Veces que se repite cada episodio
        self.episodio_actual = []
        
        # Configuración según tamaño del estado
        if cant_estados == 16:
            self.epsilon_min = 0.01
            self.epsilon_decay = self.epsilon_min ** (1 / (num_episodios * 0.9))
            self.gamma = 0.99
            self.alpha_min = 0.1
        elif cant_estados == 64:
            self.epsilon_decay = 0.9994
            self.epsilon_min = 0.01
            self.gamma = 0.99
            self.alpha_min = 0.05
    
    def elegir_accion(self, estado):
        """Elige una acción usando política epsilon-greedy."""
        if np.random.rand() < self.epsilon:
            return np.random.choice(self.action_space)
        else:
            return np.argmax(self.q_table[estado])

    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        """Almacena la experiencia y aprende cuando termina el episodio."""
        # Guardar transición en el episodio actual
        self.episodio_actual.append((estado, accion, recompensa, siguiente_estado, terminado))
        
        if terminado:
            self.cont_episodios += 1
            if(self.cont_episodios % 1000 == 0):
                print(f"Epsilon: {self.epsilon}")
            episodio_completo = list(self.episodio_actual)
            # Guardar episodio completo en el buffer
            self.replay_buffer.append(episodio_completo)

            es_exitoso = recompensa > 0
            if es_exitoso:
                self.episodios_exitosos.append(episodio_completo)
            
            # Una vez para actualziar visitas
            self._actualizar_q_desde_episodio(episodio_completo, incrementar_visitas=True)
            # Repetir el episodio actual
            for _ in range(self.replay_episodios - 1):
                self._actualizar_q_desde_episodio(episodio_completo)
            
            # Replay de episodios exitosos
            if len(self.episodios_exitosos) > 0:
                num_replays_exitosos = min(5, len(self.episodios_exitosos)) # Maximo 5 episodios exitosos
                for _ in range(num_replays_exitosos):
                    episodio_replay = random.choice(self.episodios_exitosos)
                    self._actualizar_q_desde_episodio(episodio_replay)
            
            # Replay de episodios normales (menos frecuente)
            if len(self.replay_buffer) > 10:
                num_replays_normales = 2
                for _ in range(num_replays_normales):
                    episodio_replay = random.choice(self.replay_buffer)
                    self._actualizar_q_desde_episodio(episodio_replay)

            # Limpiar episodio actual
            self.episodio_actual = []
            
            # Decrementar epsilon
            if self.full_exploracion() == False and self.epsilon > self.epsilon_min:
                self.epsilon *= self.epsilon_decay
    
    def _actualizar_q_desde_episodio(self, episodio, incrementar_visitas=False):
        """Actualiza la tabla Q usando todas las transiciones de un episodio."""
        for estado, accion, recompensa, siguiente_estado, terminado in episodio:
            # Calcular Q actual
            qActual = self.q_table[estado, accion]
            
            if incrementar_visitas:
                self.q_visitas[estado, accion] += 1
            alpha = 1.0 / (1.0 +self.q_visitas[estado, accion])
            alpha = max(alpha, self.alpha_min)
            
            # Calcular target
            if terminado:
                maxQ = 0
            else:
                maxQ = np.max(self.q_table[siguiente_estado])
            
            # Actualizar Q-table
            self.q_table[estado, accion] = (1 - alpha) * qActual + alpha * (recompensa + self.gamma * maxQ)
    
    def mostrar_q_table(self):
        print(f"Epsilon: {self.epsilon}")
        print(f"Episodios en buffer: {len(self.replay_buffer)}")
        print("Tabla Q:")
        print(self.q_table)
    
    def mostrar_visitas(self):
        print("Visitas Q:")
        print(self.q_visitas)

    def full_exploracion(self):
        return (len(self.episodios_exitosos) == 0 and self.cont_episodios < self.umbral_exito)
        

In [ ]:
num_episodios = 3000

# Entrenar agente con Experience Replay
env = gym.make('FrozenLake-v1', map_name="4x4", is_slippery=True)
agente_replay = AgenteQLearningConReplay(
    env.action_space, 
    cant_estados=16,
    replay_size=500,      # Tamaño del buffer
    replay_episodios=5,     # Veces que se repite cada episodio
    umbral_exito=1000,
    num_episodios=num_episodios
)

print("Entrenando con Experience Replay...")
entrenar_agente(agente_replay, env, num_episodios=num_episodios, max_pasos=100)
print("Epsilon luego entrenamiento:", agente_replay.epsilon)
# Evaluar
agente_replay.epsilon = 0.0
resultados = evaluar_agente(agente_replay, env, num_episodios=1000)
imprimir_resultados_evaluacion(resultados, "Agente Q-Learning con Replay")

agente_replay.mostrar_q_table()
agente_replay.mostrar_visitas()

env.close()

Entrenando con Experience Replay...
Episodio 300: Recompensa Promedio = 0.000, Longitud Promedio = 7.0
Episodio 600: Recompensa Promedio = 0.013, Longitud Promedio = 7.9
Episodio 900: Recompensa Promedio = 0.010, Longitud Promedio = 9.2
Epsilon: 0.561418483038787
Episodio 1200: Recompensa Promedio = 0.037, Longitud Promedio = 10.8
Episodio 1500: Recompensa Promedio = 0.060, Longitud Promedio = 12.0
Episodio 1800: Recompensa Promedio = 0.107, Longitud Promedio = 14.6
Epsilon: 0.20643100759521713
Episodio 2100: Recompensa Promedio = 0.140, Longitud Promedio = 17.7
Episodio 2400: Recompensa Promedio = 0.213, Longitud Promedio = 21.7
Episodio 2700: Recompensa Promedio = 0.253, Longitud Promedio = 24.9
Epsilon: 0.07590373702362134
Episodio 3000: Recompensa Promedio = 0.283, Longitud Promedio = 25.1
Epsilon luego entrenamiento: 0.07582783328659772

Agente Q-Learning con Replay - Resultados de Evaluación:
Tasa de Victorias: 47.7%
Recompensa Promedio: 0.477
Desviación Estándar: 0.499
Total de 

In [16]:
# Entrenar agente con Experience Replay
env = gym.make('FrozenLake-v1', map_name="8x8", is_slippery=True)
agente_replay = AgenteQLearningConReplay(
    env.action_space, 
    cant_estados=64,
    replay_size=1000,      # Tamaño del buffer
    replay_episodios=3,      # Veces que se repite cada episodio
    umbral_exito=3000
)
    
print("Entrenando con Experience Replay...")
entrenar_agente(agente_replay, env, num_episodios=10000, max_pasos=200)
print("Epsilon luego entrenamiento:", agente_replay.epsilon)
# Evaluar
agente_replay.epsilon = 0.0
resultados = evaluar_agente(agente_replay, env, num_episodios=1000)
imprimir_resultados_evaluacion(resultados, "Agente Q-Learning con Replay")

agente_replay.mostrar_q_table()
agente_replay.mostrar_visitas()

env.close()

Entrenando con Experience Replay...
Epsilon: 1
Episodio 1000: Recompensa Promedio = 0.000, Longitud Promedio = 32.5
Epsilon: 0.5943088635498273
Episodio 2000: Recompensa Promedio = 0.008, Longitud Promedio = 36.9
Epsilon: 0.3261048920918024
Episodio 3000: Recompensa Promedio = 0.031, Longitud Promedio = 48.2
Epsilon: 0.17893793474828412
Episodio 4000: Recompensa Promedio = 0.098, Longitud Promedio = 54.4
Epsilon: 0.09818553866701143
Episodio 5000: Recompensa Promedio = 0.212, Longitud Promedio = 62.6
Epsilon: 0.05387566374280865
Episodio 6000: Recompensa Promedio = 0.305, Longitud Promedio = 66.4
Epsilon: 0.02956226734745618
Episodio 7000: Recompensa Promedio = 0.376, Longitud Promedio = 71.5
Epsilon: 0.016221195063032996
Episodio 8000: Recompensa Promedio = 0.447, Longitud Promedio = 71.8
Epsilon: 0.009999884933492255
Episodio 9000: Recompensa Promedio = 0.447, Longitud Promedio = 75.5
Epsilon: 0.009999884933492255
Episodio 10000: Recompensa Promedio = 0.501, Longitud Promedio = 74.2


In [ ]:
# TODO: Implementar Agente Q-Learning
class AgenteReversoQLearning(Agente):
    """Agente que usa el algoritmo Q-Learning."""
    q_table = None
    q_visitas = None

    def __init__(self, action_space, cant_estados=16):
        self.q_table = np.zeros((cant_estados, 4), dtype=np.float64)
        self.q_visitas = np.zeros((cant_estados, 4), dtype=np.int32)
        self.episodios = []
        self.action_space = action_space.n
        self.epsilon = 1 
        if cant_estados == 16:
            self.epsilon_decay = 0.9995
            self.epsilon_min = 0.01
            self.gamma = 0.99
            self.alpha_min = 0.1
        elif cant_estados == 64:
            self.epsilon_decay = 0.9998 
            self.epsilon_min = 0.02
            self.gamma = 0.99
            self.alpha_min = 0.1

    def elegir_accion(self, estado):
        """Elige una acción usando política epsilon-greedy."""
        if np.random.rand() < self.epsilon:
            return np.random.choice(self.action_space)  # Exploración
        else:
            return np.argmax(self.q_table[estado])

    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        """Actualiza la tabla Q usando la ecuación de Bellman."""
        self.episodios.append((estado, accion, recompensa))

        if terminado:
            # Tomo el ultimo elemento de la lista
            retorno_ac = 0

            for estado_accion_i in reversed(self.episodios):
                estado_i, accion_i, recompensa_i = estado_accion_i

                retorno_ac = recompensa_i + self.gamma * retorno_ac

                # Visitas
                self.q_visitas[estado_i, accion_i] += 1
                alpha = 1.0 / (1.0 + self.q_visitas[estado_i, accion_i])
                alpha = max(alpha, self.alpha_min)

                # Calcula Q actual
                qActual = self.q_table[estado_i, accion_i]

                self.q_table[estado_i, accion_i] = (1 - alpha) * qActual + alpha * (retorno_ac)
            
            self.episodios = []

            if self.epsilon > self.epsilon_min:
                self.epsilon *= self.epsilon_decay
                
    def mostrar_q_table(self):
        print(self.epsilon)
        print("Tabla Q:")
        print(self.q_table)
    
    def mostrar_visitas(self):
        print("Visitas Q:")
        print(self.q_visitas)


In [ ]:
# Entrenar agente con aprendizaje reverso
env = gym.make('FrozenLake-v1', map_name="4x4", is_slippery=True)
agente_reverso = AgenteReversoQLearning(env.action_space, cant_estados=16)

print("Entrenando con aprendizaje reverso...")
entrenar_agente(agente_reverso, env, num_episodios=80000, max_pasos=200)

# Evaluar
agente_reverso.epsilon = 0.0
resultados = evaluar_agente(agente_reverso, env, num_episodios=1000)
imprimir_resultados_evaluacion(resultados, "Agente Q-Learning Reverso")

agente_reverso.mostrar_q_table()
agente_reverso.mostrar_visitas()

env.close()

Entrenando con aprendizaje reverso...
Episodio 8000: Recompensa Promedio = 0.033, Longitud Promedio = 8.0
Episodio 16000: Recompensa Promedio = 0.035, Longitud Promedio = 6.2
Episodio 24000: Recompensa Promedio = 0.038, Longitud Promedio = 6.8
Episodio 32000: Recompensa Promedio = 0.037, Longitud Promedio = 6.2
Episodio 40000: Recompensa Promedio = 0.076, Longitud Promedio = 8.3
Episodio 48000: Recompensa Promedio = 0.063, Longitud Promedio = 7.2
Episodio 56000: Recompensa Promedio = 0.081, Longitud Promedio = 8.8
Episodio 64000: Recompensa Promedio = 0.042, Longitud Promedio = 6.9
Episodio 72000: Recompensa Promedio = 0.067, Longitud Promedio = 7.5
Episodio 80000: Recompensa Promedio = 0.054, Longitud Promedio = 8.4

Agente Q-Learning Reverso - Resultados de Evaluación:
Tasa de Victorias: 6.8%
Recompensa Promedio: 0.068
Desviación Estándar: 0.252
Total de Victorias: 68
0.0
Tabla Q:
[[5.78788153e-06 9.33478100e-06 3.07087973e-05 5.67023056e-06]
 [1.57451647e-09 1.54311668e-09 4.3541218

La siguiente celda define el agente DQN a implementar. 

In [ ]:

class DQN(nn.Module):
    """Clase auxiliar que implementa una Red Q Profunda con una capa oculta."""
    
    def __init__(self, tamano_entrada, tamano_oculto, tamano_salida):
        super(DQN, self).__init__()
        pass
    
    def forward(self, x):
        pass

class AgenteDQN(Agente):
    """Agente de Red Q Profunda."""
    
    def __init__(self):
        pass
    
    def elegir_accion(self, estado):
        """Elige acción usando política epsilon-greedy."""
        pass
    
    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        pass
    


Código para entrenar y evaluar el agente de DQN 
implementado.


In [ ]:
# Ejecución del Agente DQN Determinista
env = gym.make('FrozenLake-v1')
agente_dqn = AgenteDQN(env.action_space)
entrenar_agente(agente_qlearning, env)
resultados = evaluar_agente(agente_qlearning, env, num_episodios=100)
imprimir_resultados_evaluacion(resultados, "Agente DQN")
env.close()

TypeError: AgenteDQN.__init__() takes 1 positional argument but 2 were given